# FlashRAG 入门实践

**FlashRAG** 是由中国人民大学 NLP 团队开源的 RAG（检索增强生成）研究框架。
本 Notebook 基于 **官方仓库 `main` 分支**（https://github.com/RUC-NLPIR/FlashRAG）
的实际源码整理，所有 API 均已与官方代码对照验证。

| 章节 | 内容 |
|------|------|
| 一 | 安装与环境配置 |
| 二 | 架构概览 |
| 三 | 配置系统（Config / YAML） |
| 四 | 知识库构建（分块 → JSONL → 构建 FAISS 索引） |
| 五 | 检索器（Dense / BM25 / 多路混合） |
| 六 | 重排序器（CrossReranker） |
| 七 | 生成器（OpenAI API / 本地 HF / PromptTemplate） |
| 八 | Pipeline 实战（Naive / FLARE / 自定义） |
| 九 | 结果评估 |
| **十** | **完整端到端流程（一键跑通）** |
| 十一 | 总结 |

## 一、安装与环境配置

In [ ]:
# ============================================================
# 安装 FlashRAG 及核心依赖
# ------------------------------------------------------------
# flashrag      : FlashRAG 框架本体
# transformers  : HuggingFace 模型加载（检索/生成均依赖）
# faiss-cpu     : FAISS 向量索引库（CPU 版，有 GPU 可换 faiss-gpu）
# bm25s         : BM25 稀疏检索后端（FlashRAG 默认 bm25 backend）
# datasets      : HuggingFace 数据集加载工具（FlashRAG 内部使用）
# openai        : OpenAI Python SDK（兼容所有 /v1/chat/completions 端点）
# tiktoken      : OpenAI tokenizer（OpenaiGenerator 内部依赖）
# ============================================================
%pip install flashrag transformers faiss-cpu bm25s datasets openai tiktoken --quiet

In [ ]:
# ============================================================
# 验证关键依赖是否安装成功，打印各库版本号
# ============================================================
import importlib  # 动态导入模块工具，用于版本检查
import sys        # 系统信息模块，用于打印 Python 解释器版本

# 待检查的库列表，List[Tuple[str, str]] 类型
libs = [
    ("flashrag",    "FlashRAG"),
    ("transformers","Transformers"),
    ("faiss",       "FAISS"),
    ("bm25s",       "bm25s"),
    ("datasets",    "HuggingFace Datasets"),
    ("openai",      "OpenAI SDK"),
    ("tiktoken",    "tiktoken"),
]

print(f"Python 版本: {sys.version}")
print("-" * 40)
for import_name, display_name in libs:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "已安装")
        print(f"✓ {display_name:<25} {ver}")
    except ImportError:
        print(f"✗ {display_name:<25} 未安装，请重新运行上方安装命令")

## 二、FlashRAG 架构概览

FlashRAG 整体数据流（按源码实际执行路径）：

```
原始文档
   ↓ ① split_text_into_chunks()      自定义分块函数
JSONL 语料（{id, contents, title}）
   ↓ ② Index_Builder.build_index()     flashrag.retriever.index_builder.Index_Builder
索引文件（Dense: {method}_{type}.index / BM25: {save_dir}/bm25/ 目录）
   ↓
用户问题 → ③ retriever.batch_search(queries)    批量检索（实际方法名）
                ↓
            ④ reranker.rerank(queries, docs)     CrossReranker（实际类名）
               返回 (reranked_docs, scores) 元组
                ↓
            ⑤ prompt_template.get_string(       PromptTemplate（实际方法名）
                   question=q, retrieval_result=r)
                ↓
            ⑥ generator.generate(prompts)        OpenaiGenerator / HFCausalLMGenerator
                ↓
            ⑦ evaluator.evaluate(dataset)        Evaluator
```

**步骤 ③~⑦ 均由 Pipeline 一键封装**（见第八章）。

### 组件一览（均经官方源码验证）

| 类别 | 官方模块路径 | 类/函数名 |
|------|------------|----------|
| 检索器 | `flashrag.retriever` | `DenseRetriever` / `BM25Retriever` |
| 多路检索 | Config: `use_multi_retriever=True` | `MultiRetrieverRouter`（自动选择） |
| 重排序 | `flashrag.retriever` | `CrossReranker` / `BiReranker` |
| 索引构建 | `flashrag.retriever.index_builder` | `Index_Builder` |
| 生成器(API) | `flashrag.generator` | `OpenaiGenerator` |
| 生成器(本地) | `flashrag.generator` | `HFCausalLMGenerator` / `VLLMGenerator` |
| Prompt 模板 | `flashrag.prompt` | `PromptTemplate` |
| Pipeline | `flashrag.pipeline` | `SequentialPipeline` / `FLAREPipeline` / `SelfRAGPipeline` 等 |
| 评估器 | `flashrag.evaluator` | `Evaluator` |
| 配置 | `flashrag.config` | `Config`（YAML 驱动 + `basic_config.yaml` 默认值） |

## 三、配置系统

### 3.1 创建 YAML 配置文件

FlashRAG 内置 `basic_config.yaml` 提供所有参数的默认值，
自定义 YAML 只需覆盖与默认值不同的字段即可。

**重要路径规则**（来自 `config.py` 源码）：
```
dataset_path = os.path.join(data_dir, dataset_name)
```
数据集文件应放在 `{data_dir}/{dataset_name}/test.jsonl`。

In [ ]:
import os    # 操作系统接口，用于创建目录和读取环境变量
import yaml  # YAML 序列化/反序列化库，用于写入配置文件

# ============================================================
# 定义工作目录结构
# ============================================================
WORKSPACE    = "flashrag_workspace"          # str: 根工作目录
CORPUS_DIR   = f"{WORKSPACE}/corpus"        # str: 语料存放目录
INDEX_DIR    = f"{WORKSPACE}/index"         # str: 检索索引存放目录
OUTPUT_DIR   = f"{WORKSPACE}/output"        # str: 实验结果存放目录
DATASET_NAME = "mydata"                      # str: 数据集名称（影响 dataset_path 计算）
# FlashRAG Config 内部会计算：dataset_path = data_dir / dataset_name
# 所以数据集文件须放在 WORKSPACE/DATASET_NAME/test.jsonl
DATA_SUBDIR  = f"{WORKSPACE}/{DATASET_NAME}"  # str: 实际数据集目录

for d in [CORPUS_DIR, INDEX_DIR, OUTPUT_DIR, DATA_SUBDIR]:
    # os.makedirs(path, exist_ok=True) : 递归创建多级目录，exist_ok=True 时已存在不报错
    os.makedirs(d, exist_ok=True)

# os.environ.get(key, default) : 读取环境变量，不存在时返回 default
# 设置环境变量（Windows）: set DEEPSEEK_API_KEY=sk-xxxx
# 设置环境变量（Linux/Mac）: export DEEPSEEK_API_KEY=sk-xxxx
API_KEY  = os.environ.get("DEEPSEEK_API_KEY", "your-api-key-here")  # str: API 密钥
BASE_URL = "https://api.deepseek.com/v1"   # str: API 根地址（可替换为中转站）

# ============================================================
# 自定义配置字典，仅覆盖与 basic_config.yaml 默认值不同的字段
# 未覆盖的字段由 FlashRAG 内置默认值自动填充
# ============================================================
config_dict = {
    # ---- 全局设置 ----
    "save_dir":      OUTPUT_DIR,   # str: 评估结果保存目录（Config 内部会自动追加时间戳子目录）
    "gpu_id":        "0",          # str: 使用的 GPU 编号；"" 或不设置表示仅 CPU
    "seed":          2024,          # int: 随机种子，保证实验可复现
    "save_note":     "demo",        # str: 保存目录的后缀标识

    # ---- 语料与索引路径 ----
    # corpus_path : JSONL 格式，每行 {"id": "0", "contents": "文本", "title": "..."}
    "corpus_path":   f"{CORPUS_DIR}/my_corpus.jsonl",
    # index_path  : DenseRetriever 加载时要求索引文件必须已存在（需先用 Index_Builder 构建）
    # 文件名格式（来自源码）：{retrieval_method}_{faiss_type}.index（如 e5_Flat.index）
    "index_path":    f"{INDEX_DIR}/e5_Flat.index",

    # ---- 检索器设置 ----
    # retrieval_method : 检索方式名称，同时作为模型别名
    #   常用值："e5"（intfloat/e5-base-v2）、"bge"（BAAI/bge-base-en）、"bm25"
    "retrieval_method":     "e5",
    # retrieval_model_path : 编码模型路径（本地目录或 HuggingFace Hub ID）
    # 若不指定，Config 会从 basic_config.yaml 的 model2path 字典自动查找
    "retrieval_model_path": "BAAI/bge-small-zh-v1.5",   # str: 中文模型
    "retrieval_topk":       5,        # int: 检索返回的 Top-K 文档数
    "retrieval_batch_size": 256,       # int: 构建索引时编码文档的批次大小
    # bm25_backend : BM25 后端库；官方有效值为 "bm25s"（默认）或 "pyserini"（需 Java）
    # 注意："rank_bm25" 不是 FlashRAG 支持的 backend，会导致运行错误
    "bm25_backend":         "bm25s",  # str: BM25 后端，推荐 "bm25s"

    # ---- 重排序器设置（仅在 use_reranker=True 时生效）----
    "use_reranker":         False,    # bool: 是否启用重排序（Pipeline 运行时读取）
    # rerank_model_name : 重排序模型名称（作为模型别名，Config 自动在 model2path 中查找路径）
    "rerank_model_name":    "bge-reranker-base",  # str: reranker 模型别名
    # rerank_model_path : 若不设置，Config 会从 rerank_model_name 自动查找路径
    "rerank_model_path":    "BAAI/bge-reranker-base",  # str: reranker 模型路径
    "rerank_topk":          3,         # int: 重排序后保留的文档数
    "rerank_max_length":    512,        # int: reranker 输入的最大 token 数
    "rerank_batch_size":    64,         # int: reranker 推理批次大小

    # ---- 生成器设置（OpenAI 兼容接口）----
    # framework 有效值（来自 get_generator 源码）："openai" / "hf" / "vllm" / "fschat"
    "framework":            "openai",
    # generator_model : API 模型 ID（framework="openai" 时）或本地模型路径/别名（hf/vllm 时）
    "generator_model":      "deepseek-chat",
    "generator_max_input_len": 4096,   # int: Prompt 最大 token 数（超出自动截断）
    "generator_batch_size": 1,          # int: 生成批次大小（openai 框架下每次并发请求数）
    "generation_params": {
        "max_tokens":   512,            # int: 生成回答的最大 token 数
        "temperature":  0.1,            # float: 采样温度，越低越确定
    },
    # openai_setting : 直接传给 AsyncOpenAI(**openai_setting) 的参数（来自源码）
    "openai_setting": {
        "api_key":  API_KEY,            # str: API 密钥
        "base_url": BASE_URL,           # str: API 根地址
    },

    # ---- 数据集设置 ----
    # Config 源码：dataset_path = os.path.join(data_dir, dataset_name)
    # 因此数据集文件应位于：{data_dir}/{dataset_name}/test.jsonl
    "data_dir":             WORKSPACE,        # str: 数据集根目录
    "dataset_name":         DATASET_NAME,     # str: 数据集名称（作为子目录名）
    "split":                ["test"],         # List[str]: 使用的数据集分片
    "test_sample_num":      None,             # int|None: 评估采样数量，None 表示全量

    # ---- 评估指标设置 ----
    # 指标名称来自 flashrag/evaluator/metrics.py 各类的 metric_name 属性
    # 注意：FlashRAG 使用 "em"、"f1"、"acc"，而非 RAGAS 的 "answer_em"、"answer_f1"
    # 注意：FlashRAG 使用 "retrieval_recall"，而非 "retrieval_hit@k" 或 "retrieval_mrr"
    "metrics": ["em", "f1", "acc", "retrieval_recall"],  # List[str]: 启用的评估指标
    "save_metric_score":      True,   # bool: 是否将指标结果写入 metric_score.txt
    "save_intermediate_data": False,  # bool: 是否保存每条样本的详细得分
    # metric_setting : 部分指标的额外参数字典
    #   retrieval_recall_topk : int，Retrieval_Recall/Precision 计算时取前 K 个文档
    "metric_setting": {
        "retrieval_recall_topk": 5,  # int: 与 retrieval_topk 保持一致
    },
}

# yaml.dump : 将 Python 字典序列化为 YAML 格式并写入文件
config_path = f"{WORKSPACE}/config.yaml"  # str: 配置文件路径
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(config_dict, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f"配置文件已写入: {config_path}")
print(f"数据集目录   : {DATA_SUBDIR}  (存放 test.jsonl 的实际路径)")
print(f"API Key      : {'*' * 8 + API_KEY[-4:] if len(API_KEY) > 4 else '(未设置)'}")

### 3.2 加载 Config 对象

In [ ]:
# flashrag.config.Config : FlashRAG 配置中心
# 内部加载 basic_config.yaml（默认值），再与用户 YAML 合并，最终合并运行时覆盖字典
# 优先级：运行时 override_dict > 用户 YAML > basic_config.yaml 默认值
from flashrag.config import Config

# Config(config_file_path, override_dict) :
#   config_file_path : str，用户 YAML 文件路径
#   override_dict    : Dict，运行时覆盖参数；传 {} 表示完全使用 YAML 中的值
# 返回值 : Config 对象
#   支持属性访问（config.retrieval_topk）和字典访问（config["retrieval_topk"]）
config = Config(config_path, {})

print("Config 加载成功！")
print(f"  检索方法    : {config['retrieval_method']}")
print(f"  检索 Top-K  : {config['retrieval_topk']}")
print(f"  生成框架    : {config['framework']}")
print(f"  生成模型    : {config['generator_model']}")
# dataset_path 由 Config 内部计算：data_dir + "/" + dataset_name
print(f"  数据集路径  : {config['dataset_path']}")

# ---- 消融实验示例：动态覆盖 Top-K，不修改 YAML 文件 ----
config_topk3 = Config(config_path, {"retrieval_topk": 3, "rerank_topk": 2})
print(f"\n消融实验配置：Top-K={config_topk3['retrieval_topk']}, Rerank-K={config_topk3['rerank_topk']}")

## 四、知识库构建

### 4.1 文档分块与语料写入

FlashRAG 要求语料为 **JSONL 格式**，每行必须包含：
- `id`：唯一标识（str）
- `contents`：chunk 文本内容（str）

可选字段：`title` 等元数据。

In [ ]:
import re    # 正则表达式模块，用于按段落切分
import json  # JSON 序列化模块
from typing import List, Dict  # 类型注解


def split_text_into_chunks(
    text: str,
    chunk_size: int = 200,
    chunk_overlap: int = 50
) -> List[str]:
    """
    将长文本切分为固定大小的 chunk 列表（带重叠）。

    切分策略（两步）：
      1. 按双换行（段落边界）拆分成段落列表
      2. 对超长段落按滑动窗口切分，相邻 chunk 保留 chunk_overlap 字符重叠

    Parameters
    ----------
    text         : str，输入的原始长文本
    chunk_size   : int，每个 chunk 的最大字符数，默认 200
    chunk_overlap: int，相邻 chunk 的重叠字符数，默认 50

    Returns
    -------
    chunks : List[str]，切分后的 chunk 文本列表
    """
    # re.split(r'\n\s*\n', text) : 按段落边界（连续空行）拆分，返回 List[str]
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

    chunks = []   # List[str]: 最终 chunk 列表
    for para in paragraphs:
        if len(para) <= chunk_size:
            chunks.append(para)   # 短段落直接作为一个 chunk
        else:
            # step : int，滑动步长 = chunk_size - chunk_overlap（保证 >= 1）
            step = max(1, chunk_size - chunk_overlap)
            for start in range(0, len(para), step):
                chunk = para[start:start + chunk_size]
                if chunk.strip():
                    chunks.append(chunk)

    return chunks  # List[str]，长度取决于文本长度和 chunk_size


# ============================================================
# 示例语料：关于 RAG 技术的中文知识片段
# ============================================================
# raw_texts : List[Dict]，每个 Dict 含 title 和 content 字段
raw_texts = [
    {
        "title": "RAG 技术综述",
        "content": (
            "检索增强生成（RAG）将信息检索与大语言模型结合，解决幻觉和知识更新问题。\n\n"
            "RAG 三阶段：朴素 RAG（仅检索生成）、高级 RAG（加入重排序/查询改写）、"
            "模块化 RAG（各组件可独立替换）。"
        )
    },
    {
        "title": "检索技术",
        "content": (
            "向量检索（Dense）通过编码模型将文本映射为稠密向量，利用余弦相似度或内积检索。\n\n"
            "BM25 是经典稀疏检索算法，对关键词匹配效果好，但对同义词召回弱于稠密检索。\n\n"
            "混合检索（Hybrid）用 RRF 融合两种结果，兼顾语义和关键词匹配。"
        )
    },
    {
        "title": "FlashRAG 框架",
        "content": (
            "FlashRAG 是中国人民大学开源 RAG 框架，内置 Naive RAG、Self-RAG、FLARE 等 20+ 算法，"
            "支持 DenseRetriever、BM25Retriever，"
            "以及 OpenaiGenerator、HFCausalLMGenerator 三种生成器后端。\n\n"
            "所有参数由 YAML 配置文件集中管理，通过 basic_config.yaml 提供默认值，"
            "切换算法只需修改配置，无需改动代码。"
        )
    },
    {
        "title": "高级 RAG 算法",
        "content": (
            "FLARE（Forward-Looking Active REtrieval）在生成过程中遇到低置信度 token 时"
            "主动发起新一轮检索，将检索和生成交织进行，适合多跳推理。\n\n"
            "Self-RAG 让模型生成反思 token（ISREL/ISSUP/ISUSE）"
            "自主判断是否需要检索，避免对所有问题都强制检索的低效。"
        )
    },
]

# ---- 分块并写入 JSONL 语料文件 ----
corpus_path = f"{CORPUS_DIR}/my_corpus.jsonl"  # str: 语料文件路径
total_chunks = 0  # int: chunk 计数器

with open(corpus_path, "w", encoding="utf-8") as f:
    for doc_idx, doc in enumerate(raw_texts):
        # split_text_into_chunks 返回 List[str]，每个元素为一个 chunk 文本
        chunks = split_text_into_chunks(doc["content"], chunk_size=200, chunk_overlap=30)
        for chunk_idx, chunk_text in enumerate(chunks):
            record = {
                "id":       f"{doc_idx}_{chunk_idx}",   # str: 全局唯一 id
                "title":    doc["title"],                # str: 来源文档标题
                "contents": chunk_text,                  # str: chunk 正文（必填字段）
            }
            # json.dumps + "\n" : 每条文档占一行，符合 JSONL 规范
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            total_chunks += 1

print(f"语料文件: {corpus_path}")
print(f"原始文档: {len(raw_texts)} 篇  →  chunk 总数: {total_chunks}")

### 4.2 构建 FAISS 检索索引

> **注意（来自官方源码）**：`DenseRetriever.load_index()` 在索引文件不存在时直接报错，
> **不会自动构建索引**。必须先用 `Index_Builder` 显式构建并保存索引文件，
> 再初始化 `DenseRetriever`。

In [ ]:
# flashrag.retriever.index_builder.Index_Builder :
# 负责将 JSONL 语料编码为向量并构建 FAISS 索引文件（或 BM25 索引目录）
from flashrag.retriever.index_builder import Index_Builder

# Index_Builder.__init__ 关键参数（来自源码）：
#   retrieval_method : str，检索方式名称（"e5"、"bge" 等，用于选择 encoder 类型）
#   model_path       : str，编码模型路径或 HuggingFace Hub 模型 ID
#   corpus_path      : str，JSONL 语料文件路径
#   save_dir         : str，索引文件保存目录
#   max_length       : int，编码时的最大 token 数
#   batch_size       : int，编码时的批次大小
#   use_fp16         : bool，是否使用 fp16 加速编码
#   faiss_type       : str|None，FAISS 索引类型；"Flat"=精确，"IVF"=近似（速度快）
#   bm25_backend     : str，BM25 后端，"bm25s" 或 "pyserini"
index_builder = Index_Builder(
    retrieval_method = config["retrieval_method"],    # str: "e5"
    model_path       = config["retrieval_model_path"],  # str: 编码模型路径
    corpus_path      = config["corpus_path"],         # str: JSONL 语料路径
    save_dir         = INDEX_DIR,                     # str: 索引保存目录
    max_length       = config["retrieval_query_max_length"],  # int: 最大 token 数
    batch_size       = config["retrieval_batch_size"],  # int: 编码批次大小
    use_fp16         = config["retrieval_use_fp16"],    # bool: 是否使用 fp16
    faiss_type       = "Flat",                        # str: Flat 为精确检索
)

print("开始构建 FAISS 索引（首次运行需下载模型，耗时较长）...")
# index_builder.build_index() : 执行编码 + 构建 + 保存索引
# 来自源码 build_dense_index()：
#   index_save_path = os.path.join(save_dir, f"{retrieval_method}_{faiss_type}.index")
# 即生成文件名格式为 <method>_<faiss_type>.index（如 e5_Flat.index），扩展名是 .index 不是 .faiss
# 同时生成 emb_{retrieval_method}.memmap（中间嵌入文件，可删除节省空间）
index_builder.build_index()

print(f"\n索引构建完成！索引文件保存在: {INDEX_DIR}")
# 构建完成后，更新 config 的 index_path 指向实际文件
# 来自源码 Index_Builder.build_dense_index()：
#   index_save_path = os.path.join(save_dir, f"{retrieval_method}_{faiss_type}.index")
# 即文件名格式为 <retrieval_method>_<faiss_type>.index（如 e5_Flat.index）
# 注意：扩展名是 .index，不是 .faiss
import glob  # 文件通配符查找模块
# glob.glob(pattern) : 查找匹配 pattern 的所有文件路径，返回 List[str]
index_files = glob.glob(f"{INDEX_DIR}/*.index")  # 匹配 .index 文件，不是 .faiss
if index_files:
    actual_index_path = index_files[0]  # str: 实际生成的索引文件路径（如 e5_Flat.index）
    print(f"发现索引文件: {actual_index_path}")
else:
    actual_index_path = config["index_path"]  # fallback
    print("未找到 .index 文件，请检查 Index_Builder 是否执行成功")

### 4.3 初始化 DenseRetriever

In [ ]:
# flashrag.retriever.DenseRetriever : 稠密向量检索器
# 初始化时调用 load_index()，要求 config["index_path"] 指向的文件必须已存在
from flashrag.retriever import DenseRetriever

# 用实际的索引文件路径更新 Config（覆盖 YAML 中的占位路径）
config_with_index = Config(config_path, {"index_path": actual_index_path})

# DenseRetriever(config) : 加载编码模型 + 加载 FAISS 索引
# 关键 Config 字段（来自源码 update_additional_setting）：
#   retrieval_model_path      : str，编码模型路径
#   corpus_path               : str，JSONL 语料路径（用于 id→文本 映射）
#   index_path                : str，FAISS 索引文件路径
#   retrieval_batch_size      : int，编码批次大小
#   retrieval_topk            : int，检索返回的 Top-K 文档数
#   use_reranker              : bool，是否在检索后自动重排序
#   retrieval_query_max_length: int，查询最大 token 数
#   retrieval_pooling_method  : str，向量池化方式（"mean" / "cls"）
#   use_sentence_transformer  : bool，是否使用 SentenceTransformer 编码器
retriever = DenseRetriever(config_with_index)

print(f"DenseRetriever 初始化完成！")
print(f"  索引路径: {config_with_index['index_path']}")
print(f"  Top-K   : {config_with_index['retrieval_topk']}")

## 五、检索器

### 5.1 DenseRetriever — 稠密向量检索

> **注意（来自官方源码）**：检索器的批量检索方法名为 `batch_search()`，
> 单条检索方法名为 `search()`。请勿使用 `retrieve()`（不存在）。

In [ ]:
# ============================================================
# DenseRetriever 批量检索
# ============================================================

# test_queries : List[str]，批量查询文本列表，长度 N
test_queries = [
    "RAG 技术的核心思想是什么？",
    "FlashRAG 框架支持哪些生成器后端？",
    "FLARE 算法的检索策略",
]

# retriever.batch_search(query, num, return_score) : 批量向量检索
# 参数（来自源码 batch_search 装饰器链 cache_manager → rerank_manager → _batch_search）：
#   query        : List[str]，查询文本列表，长度 N
#   num          : int|None，返回文档数，None 时使用 config["retrieval_topk"]
#   return_score : bool，是否同时返回相似度分数，默认 False
# 返回值（return_score=False 时）:
#   results : List[List[Dict]]，shape: (N, topk)
#     外层 List 长度 = N（查询数量）
#     内层 List 长度 = topk（Top-K 文档数）
#     每个 Dict 包含 "id"（str）、"contents"（str）、"title"（str，若有）
#     注意：score 字段不在 Dict 中（除非 use_reranker 使得 rerank_manager 注入分数）
results = retriever.batch_search(test_queries)

for query, docs in zip(test_queries, results):
    print(f"\n{'='*55}")
    print(f"查询: {query}")
    print(f"检索到 {len(docs)} 条文档:")
    for rank, doc in enumerate(docs, 1):
        # doc["contents"] : str，chunk 正文；doc["id"] : str，文档 id
        print(f"  [{rank}] id={doc['id']:10s} | {doc['contents'][:55]}...")

In [ ]:
# retriever.search(query, num, return_score) : 单条检索（非批量）
# 参数:
#   query        : str，单条查询文本
#   num          : int|None，返回文档数，None 时使用 retrieval_topk
#   return_score : bool，是否返回分数，默认 False
# 返回值（return_score=False）: List[Dict]，长度 = topk
single_result = retriever.search("检索增强生成")

print("单条检索结果:")
for rank, doc in enumerate(single_result, 1):
    print(f"  [{rank}] {doc['contents'][:60]}...")

### 5.2 BM25Retriever — 稀疏关键词检索

> **注意 1**：FlashRAG 官方 BM25 后端只支持 `"bm25s"` 和 `"pyserini"`。
> `"rank_bm25"` 不是有效的 backend 值，会导致运行错误。

> **注意 2（来自官方源码）**：`BM25Retriever` 初始化时调用 `bm25s.BM25.load(index_path)`，
> 与 `DenseRetriever` 一样，**不会自动构建索引**，必须先用 `Index_Builder` 构建。
> `Index_Builder` 对 BM25 会把索引保存在 `{save_dir}/bm25/` 子目录下（不是单个文件）。

In [ ]:
# flashrag.retriever.BM25Retriever : 基于 BM25 的稀疏检索器
# 无需 GPU，无需向量编码，适合精确关键词匹配
from flashrag.retriever import BM25Retriever

# ---- 第一步：先用 Index_Builder 构建 BM25 索引（与 Dense 相同，不能跳过）----
# 来自源码 build_bm25_index_bm25s()：
#   self.save_dir = os.path.join(self.save_dir, "bm25")  # 实际保存在 save_dir/bm25/ 子目录
# 所以 BM25 索引路径 = {INDEX_DIR}/bm25/
bm25_index_dir = f"{INDEX_DIR}/bm25"  # str: BM25 索引实际路径（构建后自动创建）
bm25_builder = Index_Builder(
    retrieval_method = "bm25",                  # str: 触发 BM25 索引构建分支
    model_path       = "",                       # str: BM25 不需要编码模型，传空字符串
    corpus_path      = config["corpus_path"],    # str: JSONL 语料路径
    save_dir         = INDEX_DIR,               # str: 索引保存根目录（实际生成在 INDEX_DIR/bm25/）
    max_length       = 128,                      # int: BM25 不用此参数，但接口需要传
    batch_size       = 256,                      # int: BM25 不用此参数，但接口需要传
    use_fp16         = False,                    # bool: BM25 不用此参数
    bm25_backend     = "bm25s",                 # str: 指定 bm25s 后端
)
print("构建 BM25 索引...")
bm25_builder.build_index()  # 执行构建，结果保存在 INDEX_DIR/bm25/ 目录
print(f"BM25 索引保存在: {bm25_index_dir}")

# ---- 第二步：初始化 BM25Retriever（加载已构建的索引）----
# 来自源码 BM25Retriever.load_model_corpus()：
#   self.searcher = bm25s.BM25.load(self.index_path, mmap=True, load_corpus=False)
# 即 index_path 必须指向已构建的 BM25 索引目录
config_bm25 = Config(config_path, {
    "retrieval_method": "bm25",        # str: 使用 BM25 稀疏检索
    "index_path":       bm25_index_dir, # str: 指向 Index_Builder 实际构建的目录
    # bm25_backend 有效值（来自源码 BM25Retriever.load_model_corpus）：
    #   "bm25s"    : 纯 Python 实现（推荐，默认值）
    #   "pyserini" : 基于 Java Lucene（需要 Java 环境）
    "bm25_backend":     "bm25s",        # str: BM25 后端
})

# BM25Retriever(config) : 加载预构建的 BM25 索引（不自动构建！）
bm25_retriever = BM25Retriever(config_bm25)

# batch_search(queries) : 批量 BM25 检索（方法名与 DenseRetriever 相同）
# 返回值同 DenseRetriever：List[List[Dict]]，shape: (N, topk)
bm25_results = bm25_retriever.batch_search(["BM25 稀疏检索", "FAISS 向量索引"])

print("BM25 检索结果:")
for query, docs in zip(["BM25 稀疏检索", "FAISS 向量索引"], bm25_results):
    print(f"\n  查询: {query}")
    for rank, doc in enumerate(docs, 1):
        print(f"    [{rank}] {doc['contents'][:55]}...")

### 5.3 多路混合检索（MultiRetrieverRouter）

FlashRAG 通过 `use_multi_retriever: True` 启用多路检索，
内部使用 `MultiRetrieverRouter` 管理多个子检索器。

> 注意：FlashRAG 当前版本不存在 `HybridRetriever` 独立类，
> 混合检索通过 `multi_retriever_setting` 配置项实现。

In [ ]:
# flashrag.utils.get_retriever : 根据配置自动选择并实例化检索器
# 当 config["use_multi_retriever"] = True 时，返回 MultiRetrieverRouter 实例
from flashrag.utils import get_retriever

# 多路混合检索 Config
# multi_retriever_setting 结构（来自 basic_config.yaml 和 config.py 源码）：
#   merge_method    : str，融合方式；"concat"（直接拼接）/ "rrf"（Reciprocal Rank Fusion）/ "rerank"
#   topk            : int，融合后保留的文档数（仅 rrf/rerank 模式有效）
#   retriever_list  : List[Dict]，每个子检索器的独立配置
#
# 说明（来自 Config._set_additional_key 源码）：
#   Config 在 _set_additional_key() 中会自动为每个子配置补全下列缺失字段：
#     retrieval_use_fp16, retrieval_query_max_length, faiss_gpu, retrieval_topk,
#     retrieval_batch_size, use_reranker, rerank_model_name, rerank_model_path,
#     retrieval_cache_path, save_retrieval_cache=False, use_retrieval_cache=False
#   因此不必手动填写这些字段；但在自行传 dict 给子检索器时（不经过 Config），
#   仍需手动补全。此处保留 _sub_base 仅作为明确声明，增强可读性。
_sub_base = {                         # 显式声明基础字段（Config 也会自动补全）
    "save_retrieval_cache": False,    # bool: 不保存检索缓存
    "use_retrieval_cache":  False,    # bool: 不使用检索缓存
    "retrieval_cache_path": None,     # str|None
    "use_reranker":         False,    # bool: 子检索器内部不启用 reranker
    "retrieval_model_path": None,     # str|None（Dense 需覆盖为实际路径）
}
config_multi = Config(config_path, {
    "use_multi_retriever": True,                   # bool: 启用多路检索
    "multi_retriever_setting": {
        "merge_method": "rrf",                     # str: RRF 融合排名
        "topk": 5,                                 # int: 融合后保留 Top-5
        "retriever_list": [
            {                                      # 子检索器 1：稠密检索
                **_sub_base,
                "retrieval_method":     "e5",
                "retrieval_topk":       10,         # int: 子检索器各取 Top-10 候选
                "retrieval_model_path": "BAAI/bge-small-zh-v1.5",
                # index_path 指向 Index_Builder 生成的 .index 文件
                # 格式：<retrieval_method>_<faiss_type>.index（如 e5_Flat.index）
                "index_path":           actual_index_path,
                "corpus_path":          config["corpus_path"],
            },
            {                                      # 子检索器 2：BM25 稀疏检索
                **_sub_base,
                "retrieval_method":     "bm25",
                "retrieval_topk":       10,
                # BM25 索引路径 = Index_Builder 的 {save_dir}/bm25/ 子目录
                # 来自源码 build_bm25_index_bm25s：self.save_dir = os.path.join(save_dir, "bm25")
                "index_path":           bm25_index_dir,   # str: {INDEX_DIR}/bm25/
                "corpus_path":          config["corpus_path"],
                "bm25_backend":         "bm25s",           # str: BM25 后端
            },
        ]
    },
    "index_path": actual_index_path,  # 主 retrieval_method 也需要 index_path
})

# get_retriever(config) :
# 当 use_multi_retriever=True 时，实例化 MultiRetrieverRouter
# MultiRetrieverRouter 内部同时持有所有子检索器，batch_search 时并发检索再融合
multi_retriever = get_retriever(config_multi)

# batch_search 方法在 MultiRetrieverRouter 中同样可用
multi_results = multi_retriever.batch_search(["混合检索的优势"])
# 返回值: List[List[Dict]]，shape: (N, topk)，已按 RRF 融合排名

print("多路混合检索结果（RRF 融合）:")
for rank, doc in enumerate(multi_results[0], 1):
    print(f"  [{rank}] id={doc['id']:10s} | {doc['contents'][:55]}...")

## 六、重排序器（CrossReranker）

FlashRAG 重排序器实际类名（来自 `flashrag/retriever/reranker.py` 源码）：
- `CrossReranker`：CrossEncoder 精排（输入 query+doc 对，输出相关性分数）
- `BiReranker`：双塔 BiEncoder 精排

`get_reranker(config)` 根据模型架构自动选择正确的类（推荐使用）。

In [ ]:
# flashrag.retriever.CrossReranker : 基于 CrossEncoder 的重排序器
# 实际类名来自 flashrag/retriever/reranker.py 源码（非 CrossEncoderReranker）
from flashrag.retriever import CrossReranker

# CrossReranker(config) : 初始化重排序器
# 从 Config 中读取（来自源码 BaseReranker.__init__）：
#   config["rerank_model_name"] : str，重排序模型别名
#   config["rerank_model_path"] : str，重排序模型路径
#   config["rerank_topk"]       : int，重排序后保留的文档数
#   config["rerank_max_length"] : int，输入最大 token 数
#   config["rerank_batch_size"] : int，推理批次大小
#   config["device"]            : str，设备（"cuda"/"cpu"，Config 自动设置）
config_rerank = Config(config_path, {"use_reranker": True})
reranker = CrossReranker(config_rerank)

print(f"CrossReranker 初始化完成！")
print(f"  模型路径: {config_rerank['rerank_model_path']}")
print(f"  保留 Top-K: {config_rerank['rerank_topk']}")

In [ ]:
# ============================================================
# 检索 + 重排序完整流程
# ============================================================
rerank_queries = ["RAG 与传统问答的区别", "混合检索的优势"]  # List[str]: 查询列表

# Step 1: 粗排检索，获取候选文档
# all_candidates : List[List[Dict]]，shape: (N, retrieval_topk)
all_candidates = retriever.batch_search(rerank_queries)

print("粗排候选文档数:", [len(d) for d in all_candidates])

# Step 2: 精排重排序
# reranker.rerank(query_list, doc_list, batch_size, topk) :
# 参数（来自源码 BaseReranker.rerank）：
#   query_list : str | List[str]，查询文本（支持单条 str 或列表）
#   doc_list   : List[str_or_dict] | List[List[str_or_dict]]，候选文档
#                若 query_list 为 str，doc_list 可以是 List[Dict]（自动包装）
#                若 query_list 为 List，doc_list 必须是 List[List[Dict]]
#   batch_size : int|None，推理批次大小，None 使用 config 中的值
#   topk       : int|None，保留文档数，None 使用 config 中的值
# 返回值（重要！来自源码）：
#   (final_docs, final_scores) : Tuple[List[List[Dict]], List[List[float]]]
#     final_docs   : List[List[Dict]]，重排序后的文档列表，已按相关性降序排列
#                    shape: (N, topk)
#     final_scores : List[List[float]]，对应的 CrossEncoder 分数
#                    shape: (N, topk)，值越高越相关（logit 值，无固定范围）
reranked_docs, rerank_scores = reranker.rerank(rerank_queries, all_candidates)

print("\n--- 重排序结果 ---")
for q, docs, scores in zip(rerank_queries, reranked_docs, rerank_scores):
    print(f"\n查询: {q}")
    print(f"精排后保留 {len(docs)} 条文档:")
    for rank, (doc, score) in enumerate(zip(docs, scores), 1):
        # score : float，CrossEncoder logit 分数，越大越相关
        print(f"  [{rank}] score={score:.4f} | {doc['contents'][:50]}...")

## 七、生成器

### 7.1 OpenaiGenerator — OpenAI 兼容 API 生成器

In [ ]:
# flashrag.generator.OpenaiGenerator : OpenAI 兼容 API 生成器
# 内部使用 AsyncOpenAI SDK（来自源码 openai_generator.py）
# 支持所有兼容 /v1/chat/completions 或 /v1/completions 的服务
from flashrag.generator import OpenaiGenerator

# OpenaiGenerator(config) : 使用 Config 初始化
# 关键字段（来自源码 update_base_setting）：
#   config["generator_model"]     : str，API 模型 ID
#   config["openai_setting"]      : Dict，直接传给 AsyncOpenAI(**openai_setting)
#   config["generation_params"]   : Dict，生成超参数（max_tokens、temperature 等）
#   config["generator_batch_size"]: int，并发请求数
generator = OpenaiGenerator(config_with_index)

# generator.generate(input_list, batch_size, return_scores, **params) :
# 参数（来自源码 _generate_async）：
#   input_list    : List[str | dict | List[dict]]，输入列表
#     List[str]         → completion 模式（直接生成）
#     List[List[dict]]  → chat 模式（messages 列表，含 role/content）
#   return_scores : bool，是否返回 logprobs，默认 False
# 返回值:
#   answers : List[str]，生成的回答列表，长度 = len(input_list)
#   （若 return_scores=True，返回 (List[str], List[scores])）

question = "RAG 技术的三个发展阶段是什么？"  # str: 用户问题

# 先检索相关文档
top_docs = retriever.batch_search([question])[0]  # List[Dict]: Top-K 候选文档

# 手动构造 Prompt（chat 模式，使用消息列表）
context = "\n\n".join(
    f"[文档 {i+1}] {doc['contents']}" for i, doc in enumerate(top_docs)
)
# messages : List[Dict]，OpenAI chat 格式的消息列表
messages = [
    {"role": "system",  "content": "你是一个专业的知识问答助手，请根据参考文档准确回答。"},
    {"role": "user",    "content": f"参考文档：\n{context}\n\n问题：{question}"},
]

# 传入 List[List[Dict]]（chat 模式）
answers = generator.generate([messages])
# answers : List[str]，长度 1，对应第一条消息列表的生成结果

print(f"问题: {question}")
print(f"\n回答: {answers[0]}")

### 7.2 HFCausalLMGenerator — 本地 HuggingFace 模型生成器

当 `framework="hf"` 时，`get_generator()` 自动实例化 `HFCausalLMGenerator`
（针对 CausalLM 架构的模型，如 Qwen、LLaMA 等）。

> 注意：官方类名为 `HFCausalLMGenerator`，不是 `HFGenerator`。

In [ ]:
# flashrag.utils.get_generator(config) :
# 根据 config["framework"] 自动选择正确的生成器类（推荐使用此函数而非直接导入类）
#   "openai" → OpenaiGenerator
#   "hf"     → HFCausalLMGenerator（CausalLM）或 EncoderDecoderGenerator（T5/BART）
#   "vllm"   → VLLMGenerator
#   "fschat" → FastChatGenerator
from flashrag.utils import get_generator

# 切换为本地模型只需修改 framework 和 generator_model_path
config_hf = Config(config_path, {
    "framework":            "hf",                   # str: 本地 HuggingFace 模型
    # generator_model_path : 本地模型目录路径（需提前下载）
    # 若 generator_model_path 为 None，Config 会在 model2path 中查找 generator_model 对应路径
    "generator_model_path": "/path/to/local/Qwen2.5-1.5B-Instruct",  # str: 替换为实际路径
    "generation_params": {
        # HF 模型使用 max_new_tokens（而非 max_tokens），注意区分
        "max_new_tokens": 256,           # int: 最大新生成 token 数
        "temperature":    0.1,            # float: 采样温度
        "do_sample":      True,           # bool: True=采样，False=贪婪解码
    },
})

# 以下代码在本地模型不存在时会报错，仅演示 API 用法
# 取消注释并替换 generator_model_path 后可实际运行

# hf_gen = get_generator(config_hf)     # 自动选择 HFCausalLMGenerator
# answers = hf_gen.generate([messages]) # 与 OpenaiGenerator 完全相同的调用接口

print("HFCausalLMGenerator 使用说明:")
print("  1. 修改 generator_model_path 为本地模型目录")
print("  2. 取消上方两行注释")
print("  3. Pipeline 及评估代码无需任何修改")
print(f"\n  当前 HF 配置: framework={config_hf['framework']}, model_path={config_hf['generator_model_path']}")

### 7.3 PromptTemplate — 自定义 Prompt 模板

> 注意（来自官方源码）：构造 Prompt 的方法名为 `get_string()`，
> 不是 `build_prompt()`。

In [ ]:
# flashrag.prompt.PromptTemplate : Prompt 模板管理类（来自 flashrag/prompt/base_prompt.py）
# Pipeline 内部使用此类构造 Prompt，也可单独使用
from flashrag.prompt import PromptTemplate

# PromptTemplate(config, system_prompt, user_prompt, ...) :
# 参数（来自源码）：
#   config        : Config 对象
#   system_prompt : str|None，系统角色提示（放入 messages 的 system 消息）
#   user_prompt   : str|None，用户消息模板（含 {question}、{retrieval_result} 等占位符）
template = PromptTemplate(
    config_with_index,
    system_prompt="你是一个专业的知识问答助手，请根据文档准确回答，不要编造不在文档中的信息。",
    user_prompt="参考文档：\n{reference}\n\n问题：{question}",
    # reference 占位符：Pipeline 内部将检索文档格式化为字符串后填入
)

# template.get_string(question, retrieval_result, ...) :
# 实际方法名（来自源码 SequentialPipeline.run 中的调用）
# 参数：
#   question         : str，用户问题，填充 {question} 占位符
#   retrieval_result : List[Dict]，检索结果，框架自动拼接为文档文本并填充 {reference}
# 返回值:
#   prompt : str | List[Dict]，构造好的 Prompt
#     若 system_prompt 不为 None，返回 messages 列表（chat 格式）
#     否则返回纯字符串（completion 格式）
demo_docs = retriever.batch_search(["RAG 技术"])[0][:2]  # List[Dict]: 取 Top-2
prompt_result = template.get_string(
    question         = "什么是 RAG 技术？",
    retrieval_result = demo_docs,
)

print("PromptTemplate.get_string() 构造的 Prompt 类型:", type(prompt_result))
if isinstance(prompt_result, list):
    # chat 模式：返回 List[Dict]
    for msg in prompt_result:
        # msg : Dict，含 "role" 和 "content" 字段
        print(f"  [{msg['role']}] {str(msg['content'])[:80]}...")
else:
    # completion 模式：返回 str
    print(prompt_result[:200] + "...")

## 八、Pipeline 实战

FlashRAG Pipeline 一览（来自 `flashrag/pipeline/__init__.py` 源码）：

| 类名 | 来源模块 | 算法 |
|------|---------|------|
| `SequentialPipeline` | `pipeline.py` | 标准 Naive RAG |
| `ConditionalPipeline` | `pipeline.py` | 判断是否需要检索 |
| `FLAREPipeline` | `active_pipeline.py` | 前向迭代检索 |
| `SelfRAGPipeline` | `active_pipeline.py` | 自适应检索 |
| `IterativePipeline` | `active_pipeline.py` | 多轮迭代检索 |
| `IRCOTPipeline` | `active_pipeline.py` | CoT + 迭代检索 |
| `REPLUGPipeline` | `branching_pipeline.py` | 集成多路检索 |

> 注意：当前版本不存在 `HyDEPipeline`（未在 `__init__.py` 的任何导入中列出）。
> HyDE 可通过自定义 Pipeline 手动实现（见 8.4 节）。

### 8.1 准备 Pipeline 输入数据集

In [ ]:
import json  # JSON 序列化模块

# ============================================================
# FlashRAG 数据集 JSONL 格式（每行必填字段）：
#   question       : str，问题文本
#   golden_answers : List[str]，参考答案列表（评估用）
#   id             : str，问题唯一标识（可选）
# ============================================================
# 注意（来自 Config 源码）：
#   dataset_path = data_dir / dataset_name
#   所以数据集文件须位于 {WORKSPACE}/{DATASET_NAME}/test.jsonl
# ============================================================
qa_data = [
    {"id": "q0", "question": "RAG 技术的三个发展阶段是什么？",
     "golden_answers": ["朴素 RAG、高级 RAG、模块化 RAG"]},
    {"id": "q1", "question": "FlashRAG 支持哪些生成器后端？",
     "golden_answers": ["OpenaiGenerator、HFCausalLMGenerator、VLLMGenerator"]},
    {"id": "q2", "question": "FLARE 算法的核心特点是什么？",
     "golden_answers": ["在生成过程中遇到低置信度 token 时主动触发新一轮检索"]},
]

# 保存到正确路径：{WORKSPACE}/{DATASET_NAME}/test.jsonl
# （对应 Config 中 dataset_path = data_dir / dataset_name 的计算结果）
test_data_path = f"{DATA_SUBDIR}/test.jsonl"  # str: 实际数据集文件路径
with open(test_data_path, "w", encoding="utf-8") as f:
    for item in qa_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"数据集写入: {test_data_path}，共 {len(qa_data)} 条")

# flashrag.utils.get_dataset(config) :
# 读取 config["dataset_path"]（= data_dir/dataset_name）目录下的 JSONL 文件
# 返回 Dict[str, Dataset]，key 为 split 名（"test"），value 为 Dataset 对象
from flashrag.utils import get_dataset

all_splits   = get_dataset(config_with_index)  # Dict[str, Dataset]
test_dataset = all_splits["test"]              # Dataset: 测试集

print(f"Dataset 加载完成，共 {len(test_dataset)} 条")
print(f"  第1条 question: {test_dataset[0]['question']}")

### 8.2 SequentialPipeline — Naive RAG

`SequentialPipeline.run()` 内部调用 `retriever.batch_search()`（来自源码），
再调用 `prompt_template.get_string()` 构造 Prompt，最后调用 `generator.generate()`。

In [ ]:
# flashrag.pipeline.SequentialPipeline : 标准 Naive RAG Pipeline
# 流程（来自源码 SequentialPipeline.run）：
#   retriever.batch_search(questions) → prompt_template.get_string() → generator.generate()
from flashrag.pipeline import SequentialPipeline

# SequentialPipeline(config, prompt_template, retriever, generator) :
# 参数（来自源码，均有默认值）：
#   config          : Config，必填
#   prompt_template : PromptTemplate|None，None 时使用 PromptTemplate(config)
#   retriever       : Retriever|None，None 时用 get_retriever(config) 自动创建
#   generator       : Generator|None，None 时用 get_generator(config) 自动创建
naive_pipeline = SequentialPipeline(config_with_index)

# pipeline.run(dataset, do_eval, pred_process_fun) :
#   dataset         : Dataset，输入数据集
#   do_eval         : bool，是否计算评估指标，默认 True
#   pred_process_fun: callable|None，后处理函数
# 返回值:
#   output_dataset  : Dataset，增加了以下字段：
#     "pred"             : str，模型预测答案
#     "retrieval_result" : List[Dict]，检索结果
#     "prompt"           : str|List[Dict]，实际使用的 Prompt
print("运行 Naive RAG Pipeline...")
naive_output = naive_pipeline.run(test_dataset, do_eval=False)

print("\n--- Naive RAG 结果 ---")
for item in naive_output:
    print(f"\n问题: {item['question']}")
    print(f"预测: {item['pred'][:80]}...")

### 8.3 FLAREPipeline — 前向迭代检索

In [ ]:
# flashrag.pipeline.FLAREPipeline : 前向迭代检索 Pipeline
# 来自 flashrag/pipeline/active_pipeline.py
# 流程：生成 → 监控 token 置信度（logprobs）→ 低置信时主动检索 → 融入上下文继续生成
from flashrag.pipeline import FLAREPipeline

# FLARE 需要 logprobs，需在 generation_params 中启用
config_flare = Config(config_path, {
    "index_path": actual_index_path,
    "generation_params": {
        "max_tokens":  512,
        "temperature": 0.1,
        "logprobs":    True,    # bool: 开启 token 概率输出（FLARE 必须启用）
        "top_logprobs": 1,      # int: 每个位置返回 Top-1 logprob
    },
})

# FLAREPipeline(config, ...) : 初始化，参数同 SequentialPipeline
flare_pipeline = FLAREPipeline(config_flare)

print("运行 FLARE Pipeline（迭代检索）...")
flare_output = flare_pipeline.run(test_dataset, do_eval=False)

print("\n--- FLARE 结果 ---")
for item in flare_output:
    print(f"\n问题: {item['question']}")
    print(f"预测: {item['pred'][:80]}...")

### 8.4 手动实现 HyDE 逻辑

HyDE（Hypothetical Document Embeddings）在 FlashRAG 当前版本中没有独立 Pipeline 类，
可通过两步调用手动实现：
1. 用生成器生成假设文档（Hypothetical Document）
2. 用假设文档的向量代替问题向量进行检索

In [ ]:
# ============================================================
# HyDE 手动实现（两步）
# 来自 HyDE 论文（Gao et al. 2022）思路：
#   Step 1: Query → Generator → Hypothetical Document
#   Step 2: Hypothetical Document → Retriever → Top-K 真实文档
#   Step 3: 真实文档 → Generator → 最终回答
# ============================================================

hyde_question = "什么是 RAG 技术？"  # str: 用户问题

# ---- Step 1: 生成假设文档 ----
# 用 generator 根据问题生成一段假设性的回答文档
hyde_prompt_messages = [
    {"role": "user", "content":
     f"请写一段关于以下问题的回答段落（不要直接回答问题，而是写成百科知识文档风格）：\n{hyde_question}"
    }
]

# generator.generate([messages]) : 传入 List[List[Dict]]（chat 模式）
# 返回值 : List[str]，长度 1，第一条为假设文档文本
hypo_doc_texts = generator.generate([hyde_prompt_messages])
hypo_doc = hypo_doc_texts[0]  # str: 生成的假设文档文本

print(f"假设文档（前 100 字）: {hypo_doc[:100]}...")

# ---- Step 2: 用假设文档检索真实文档 ----
# 直接将假设文档作为查询传入检索器
# retriever.batch_search([hypo_doc]) : 以假设文档作为查询文本进行向量检索
# 返回值 : List[List[Dict]]，取第 0 个（对应唯一一条查询）
hyde_retrieved = retriever.batch_search([hypo_doc])[0]  # List[Dict]: Top-K 真实文档

# ---- Step 3: 基于真实文档生成最终回答 ----
context = "\n\n".join(
    f"[文档 {i+1}] {doc['contents']}" for i, doc in enumerate(hyde_retrieved)
)
final_messages = [
    {"role": "system", "content": "请根据参考文档准确回答问题。"},
    {"role": "user",   "content": f"参考文档：\n{context}\n\n问题：{hyde_question}"},
]
final_answers = generator.generate([final_messages])

print(f"\n问题: {hyde_question}")
print(f"检索文档数: {len(hyde_retrieved)}")
print(f"最终回答: {final_answers[0][:120]}...")

### 8.5 自定义 Pipeline

继承 `BasicPipeline`，重写 `run()` 方法，实现任意检索-生成逻辑。

In [ ]:
# flashrag.pipeline.BasicPipeline : 所有 Pipeline 的基类（来自 pipeline.py 源码）
# 提供 self.retriever、self.generator、self.evaluator、self.prompt_template
from flashrag.pipeline import BasicPipeline


class RerankRAGPipeline(BasicPipeline):
    """
    带 CrossReranker 精排的自定义 RAG Pipeline。

    流程：
      1. batch_search() 粗排，召回 Top-K 候选文档
      2. CrossReranker.rerank() 精排，筛选 Top-N 文档
      3. get_string() 构造 Prompt，generate() 生成最终回答

    Parameters
    ----------
    config   : Config，FlashRAG 配置对象
    reranker : CrossReranker，已初始化的重排序器
    """

    def __init__(self, config, reranker):
        """
        初始化自定义 Pipeline，注入外部重排序器。

        Parameters
        ----------
        config   : Config，配置对象
        reranker : CrossReranker，重排序器实例
        """
        # super().__init__ 自动初始化 self.retriever、self.generator 等（来自 BasicPipeline 源码）
        # BasicPipeline(config, prompt_template=None)：两个参数
        super().__init__(config)
        self.reranker = reranker  # CrossReranker: 保存重排序器

    def run(self, dataset, do_eval: bool = True, pred_process_fun=None):
        """
        执行完整「检索 → 重排序 → 生成」流程。

        Parameters
        ----------
        dataset         : Dataset，FlashRAG 格式的问答数据集
        do_eval         : bool，是否计算评估指标，默认 True
        pred_process_fun: callable|None，预测结果后处理函数

        Returns
        -------
        dataset : Dataset，增加了 pred 和 retrieval_result 字段
        """
        # dataset.question : List[str]，数据集所有问题的属性访问（来自 Dataset 源码）
        questions = dataset.question  # List[str]: 所有问题，长度 N

        # ---- Step 1: 批量粗排检索 ----
        # self.retriever.batch_search(questions) : 返回 List[List[Dict]]，shape (N, topk)
        all_candidates = self.retriever.batch_search(questions)

        # ---- Step 2: 批量精排重排序 ----
        # self.reranker.rerank(query_list, doc_list) :
        #   返回 (final_docs, final_scores)，final_docs shape: (N, rerank_topk)
        all_reranked, all_scores = self.reranker.rerank(questions, all_candidates)

        # ---- Step 3: 构造 Prompt 并生成 ----
        # self.prompt_template.get_string(question, retrieval_result) : 构造 Prompt
        input_prompts = [
            self.prompt_template.get_string(question=q, retrieval_result=docs)
            for q, docs in zip(questions, all_reranked)
        ]  # List[str | List[Dict]]: 每条问题对应一个 Prompt，长度 N

        # self.generator.generate(input_prompts) : 批量生成
        # 返回 List[str]，长度 N
        preds = self.generator.generate(input_prompts)

        # ---- 写回 Dataset（来自官方 Pipeline 源码的惯用方式）----
        # dataset.update_output(field, values) : 向数据集添加新字段
        #   field  : str，字段名
        #   values : List，与数据集长度相同的列表
        dataset.update_output("pred", preds)
        dataset.update_output("retrieval_result", all_reranked)

        # ---- 评估（调用父类方法）----
        # self.evaluate(dataset, do_eval, pred_process_fun) : 来自 BasicPipeline 源码
        dataset = self.evaluate(dataset, do_eval=do_eval, pred_process_fun=pred_process_fun)
        return dataset  # Dataset，含 pred 和 retrieval_result 字段


# ---- 使用自定义 Pipeline ----
reranker_config = Config(config_path, {"index_path": actual_index_path, "use_reranker": True})
custom_reranker  = CrossReranker(reranker_config)
custom_pipeline  = RerankRAGPipeline(reranker_config, custom_reranker)

print("运行自定义 RerankRAG Pipeline...")
custom_output = custom_pipeline.run(test_dataset, do_eval=False)

print("\n--- 自定义 Pipeline 结果 ---")
for item in custom_output:
    print(f"\n问题   : {item['question']}")
    print(f"精排数 : {len(item['retrieval_result'])}")
    print(f"预测   : {item['pred'][:80]}...")

## 九、结果评估

FlashRAG Evaluator 的指标名称来自 `flashrag/evaluator/metrics.py` 源码，
每个指标类的 `metric_name` 属性决定配置中填写的名称，
`calculate_metric()` 返回的字典 key 决定 `evaluate()` 输出的实际 key。

| 配置中填写（`metrics` 列表） | `evaluate()` 输出的 key | 指标含义 |
|--------------------------|------------------------|----------|
| `"em"` | `em` | 预测与参考答案完全匹配（Exact Match） |
| `"f1"` | `f1` | 词级别 F1（精确率与召回率的调和平均） |
| `"acc"` | `acc` | 答案包含率（预测文本含参考答案则得 1） |
| `"precision"` | `precision` | 词级别精确率 |
| `"recall"` | `recall` | 词级别召回率 |
| `"retrieval_recall"` | `retrieval_recall_top{k}` | 检索召回率（Top-K 文档是否含答案） |
| `"retrieval_precision"` | `retrieval_precision_top{k}` | 检索精确率（Top-K 中含答案的比例） |
| `"rouge-1"` / `"rouge-2"` / `"rouge-l"` | `rouge-1` 等 | ROUGE 系列（需安装 `rouge` 库） |
| `"zh_rouge-1"` 等 | `zh_rouge-1` 等 | 中文 ROUGE（需 `rouge_chinese` + `jieba`） |
| `"bleu"` | `bleu` | BLEU 分数 |
| `"input_tokens"` | `avg_input_tokens` | 平均输入 token 数 |
| `"llm_judge"` | `llm_judge_score` | LLM 打分（0~1，需额外配置） |

> **常见误区**：FlashRAG **没有** `answer_em`、`answer_f1`、`retrieval_hit@k`、`retrieval_mrr`
> 这几个指标名，这些是 **RAGAS** 框架的指标名称，不要混淆。

In [ ]:
# flashrag.evaluator.Evaluator : 内置评估器（来自 flashrag/evaluator/evaluator.py）
# __init__ 时读取 config["metrics"] 列表，并为每个指标名实例化对应的 Metric 类
# 若指标名不在 BaseMetric 子类的 metric_name 中，会抛出 NotImplementedError
from flashrag.evaluator import Evaluator

# Evaluator(config) : 初始化评估器
evaluator = Evaluator(config_with_index)

# evaluator.evaluate(dataset) : 计算评估指标
# 参数:
#   dataset : Dataset，必须含 pred（预测）和 golden_answers（参考答案）字段
# 返回值:
#   eval_result : Dict[str, float]，key=指标名，value=分数 [0,1]

print("=" * 55)
print("各 Pipeline 评估结果对比")
print("=" * 55)

pipeline_outputs = [
    ("Naive RAG",   naive_output),
    ("FLARE RAG",   flare_output),
    ("Rerank RAG",  custom_output),
]

# all_evals : Dict[str, Dict[str, float]]，各 Pipeline 的评估结果汇总
all_evals = {}
for name, output in pipeline_outputs:
    scores = evaluator.evaluate(output)  # Dict[str, float]: 评估分数
    all_evals[name] = scores

# 打印对比表格
all_metrics = sorted({m for s in all_evals.values() for m in s})  # List[str]
header = f"{'Pipeline':<15}" + "".join(f"{m:>10}" for m in all_metrics)
print(header)
print("-" * len(header))
for name, scores in all_evals.items():
    row = f"{name:<15}"
    for m in all_metrics:
        # scores.get(m, 0.0) * 100 : float，转为百分比
        row += f"{scores.get(m, 0.0) * 100:>9.1f}%"
    print(row)

## 十、完整端到端流程

将前九章所有组件串联为 6 个函数，清晰展示各步骤的拼接方式：

```
原始文档
   ↓ step1_build_corpus()       分块 → 写入 JSONL
   ↓ step2_build_index()        Index_Builder → FAISS 索引文件
   ↓ step3_setup_config()       创建 YAML → 加载 Config
   ↓ step4_prepare_dataset()    问答对 → Dataset 对象
   ↓ step5_run_pipeline()       batch_search → rerank → get_string → generate
   ↓ step6_evaluate()           Evaluator → 保存 JSON 报告
评估报告
```

### 10.1 端到端封装函数

In [ ]:
import os
import re
import glob
import json
import yaml
from typing import List, Dict, Optional
from datetime import datetime

from flashrag.config              import Config
from flashrag.retriever           import DenseRetriever, CrossReranker
from flashrag.retriever.index_builder import Index_Builder
from flashrag.generator           import OpenaiGenerator
from flashrag.pipeline            import BasicPipeline
from flashrag.evaluator           import Evaluator
from flashrag.utils               import get_dataset
from flashrag.prompt              import PromptTemplate


def step1_build_corpus(
    raw_texts: List[Dict],
    corpus_path: str,
    chunk_size: int = 200,
    chunk_overlap: int = 30
) -> int:
    """
    将原始文档列表分块并写入 JSONL 语料文件。

    Parameters
    ----------
    raw_texts    : List[Dict]，每个 Dict 含 title 和 content 字段
    corpus_path  : str，输出 JSONL 文件路径
    chunk_size   : int，每个 chunk 最大字符数
    chunk_overlap: int，相邻 chunk 重叠字符数

    Returns
    -------
    total_chunks : int，写入的 chunk 总数
    """
    os.makedirs(os.path.dirname(corpus_path), exist_ok=True)
    total_chunks = 0  # int: chunk 计数器

    with open(corpus_path, "w", encoding="utf-8") as f:
        for doc_idx, doc in enumerate(raw_texts):
            paragraphs = [p.strip() for p in re.split(r'\n\s*\n', doc["content"]) if p.strip()]
            step = max(1, chunk_size - chunk_overlap)  # int: 滑动步长
            for para in paragraphs:
                sub_chunks = (
                    [para] if len(para) <= chunk_size
                    else [para[s:s + chunk_size] for s in range(0, len(para), step)]
                )
                for chunk in sub_chunks:
                    if chunk.strip():
                        f.write(json.dumps({
                            "id":       f"{doc_idx}_{total_chunks}",  # str: 全局唯一 id
                            "title":    doc.get("title", ""),           # str: 文档标题
                            "contents": chunk,                          # str: chunk 正文
                        }, ensure_ascii=False) + "\n")
                        total_chunks += 1

    return total_chunks  # int: 总 chunk 数


def step2_build_index(
    corpus_path: str,
    index_dir: str,
    model_path: str = "BAAI/bge-small-zh-v1.5",
    retrieval_method: str = "e5",
) -> str:
    """
    使用 Index_Builder 构建 FAISS 索引文件。

    Parameters
    ----------
    corpus_path      : str，JSONL 语料文件路径
    index_dir        : str，索引保存目录
    model_path       : str，编码模型路径/Hub ID
    retrieval_method : str，检索方式名称（决定编码器类型）

    Returns
    -------
    index_path : str，生成的 FAISS 索引文件路径
    """
    # Index_Builder 参数（来自源码 __init__）
    builder = Index_Builder(
        retrieval_method = retrieval_method,  # str: 检索方式
        model_path       = model_path,         # str: 编码模型路径
        corpus_path      = corpus_path,        # str: JSONL 语料
        save_dir         = index_dir,          # str: 索引保存目录
        max_length       = 128,                # int: 编码最大长度
        batch_size       = 256,                # int: 编码批次大小
        use_fp16         = True,               # bool: 使用 fp16
        faiss_type       = "Flat",             # str: 精确检索类型
    )
    builder.build_index()  # 执行构建，在 save_dir 中生成 .index 文件

    # 查找生成的索引文件
    # 来自源码：文件名格式 = {retrieval_method}_{faiss_type}.index（如 e5_Flat.index）
    index_files = glob.glob(f"{index_dir}/*.index")  # 注意：扩展名 .index，不是 .faiss
    assert index_files, f"索引构建失败，{index_dir} 中未找到 .index 文件"
    return index_files[0]  # str: 索引文件路径


def step3_setup_config(
    workspace: str,
    dataset_name: str,
    corpus_path: str,
    index_path: str,
    api_key: str,
    base_url: str,
    generator_model: str = "deepseek-chat",
    retrieval_model: str = "BAAI/bge-small-zh-v1.5",
    rerank_model: str = "BAAI/bge-reranker-base",
    retrieval_topk: int = 5,
    rerank_topk: int = 3,
) -> Config:
    """
    创建 YAML 配置文件并返回加载后的 Config 对象。

    Returns
    -------
    config : Config，加载完毕的 FlashRAG 配置对象
    """
    os.makedirs(workspace, exist_ok=True)

    cfg = {
        "save_dir":             f"{workspace}/output",
        "gpu_id":               "0",
        "seed":                 2024,
        "save_note":            "e2e",
        "corpus_path":          corpus_path,                 # str: JSONL 语料
        "index_path":           index_path,                  # str: FAISS 索引文件（已存在）
        "retrieval_method":     "e5",
        "retrieval_model_path": retrieval_model,
        "retrieval_topk":       retrieval_topk,
        "retrieval_batch_size": 256,
        "bm25_backend":         "bm25s",                     # str: 有效值 "bm25s" / "pyserini"
        "use_reranker":         True,                        # bool: 启用重排序
        "rerank_model_name":    "bge-reranker-base",
        "rerank_model_path":    rerank_model,
        "rerank_topk":          rerank_topk,
        "rerank_max_length":    512,
        "rerank_batch_size":    64,
        "framework":            "openai",
        "generator_model":      generator_model,
        "generator_max_input_len": 4096,
        "generator_batch_size": 1,
        "generation_params":    {"max_tokens": 512, "temperature": 0.1},
        "openai_setting":       {"api_key": api_key, "base_url": base_url},
        # dataset_path = data_dir / dataset_name（Config 自动计算）
        "data_dir":             workspace,                   # str: 数据集根目录
        "dataset_name":         dataset_name,                # str: 数据集名（子目录）
        "split":                ["test"],
        "test_sample_num":      None,
    }

    config_path = f"{workspace}/config.yaml"  # str: 配置文件路径
    os.makedirs(workspace, exist_ok=True)
    with open(config_path, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

    return Config(config_path, {})  # Config 对象


def step4_prepare_dataset(
    qa_pairs: List[Dict],
    workspace: str,
    dataset_name: str,
    config: Config
):
    """
    将问答对写入正确路径的 JSONL 文件，并加载为 FlashRAG Dataset。

    路径规则（来自 Config 源码）：
      dataset_path = data_dir / dataset_name
      文件须放在 dataset_path/test.jsonl

    Returns
    -------
    dataset : Dataset，FlashRAG 格式数据集
    """
    # 数据集目录 = workspace / dataset_name
    data_subdir = os.path.join(workspace, dataset_name)  # str: 数据集子目录
    os.makedirs(data_subdir, exist_ok=True)

    jsonl_path = os.path.join(data_subdir, "test.jsonl")  # str: JSONL 文件路径
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for item in qa_pairs:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    return get_dataset(config)["test"]  # Dataset 对象


def step5_run_pipeline(config: Config, dataset, reranker: CrossReranker):
    """
    执行「检索 → 重排序 → 生成」完整流程。

    使用正确的 API：
    - retriever.batch_search() （非 retrieve()）
    - reranker.rerank() 返回 (docs, scores) 元组
    - prompt_template.get_string() （非 build_prompt()）
    - generator.generate()

    Returns
    -------
    dataset : Dataset，增加了 pred 和 retrieval_result 字段
    """
    retriever = DenseRetriever(config)   # DenseRetriever: 加载已有 FAISS 索引
    prompt_template = PromptTemplate(config)  # PromptTemplate: 使用默认模板
    generator_obj = OpenaiGenerator(config)   # OpenaiGenerator: API 生成器

    questions = dataset.question  # List[str]: 数据集所有问题

    print("  [1/3] 批量粗排检索...")
    # batch_search 返回 List[List[Dict]]，shape: (N, retrieval_topk)
    all_candidates = retriever.batch_search(questions)

    print("  [2/3] 精排重排序...")
    # rerank 返回 (List[List[Dict]], List[List[float]])，两个长度 N 的列表
    all_reranked, _ = reranker.rerank(questions, all_candidates)

    print("  [3/3] 构造 Prompt 并生成...")
    # get_string 返回 str 或 List[Dict]（取决于是否设置了 system_prompt）
    input_prompts = [
        prompt_template.get_string(question=q, retrieval_result=docs)
        for q, docs in zip(questions, all_reranked)
    ]  # List[str | List[Dict]]，长度 N

    preds = generator_obj.generate(input_prompts)  # List[str]，长度 N

    dataset.update_output("pred", preds)
    dataset.update_output("retrieval_result", all_reranked)
    return dataset  # Dataset，含 pred 和 retrieval_result 字段


def step6_evaluate(config: Config, output_dataset, extra_info: Optional[Dict] = None) -> Dict:
    """
    评估 Pipeline 输出并保存 JSON 报告。

    Returns
    -------
    report : Dict，含所有评估指标分数的报告字典
    """
    # Evaluator.evaluate 返回 Dict[str, float]，key=指标名，value∈[0,1]
    scores = Evaluator(config).evaluate(output_dataset)  # Dict[str, float]

    # 注意（来自 config.py 源码）：
    #   Config.__getitem__ 使用 final_config.get(item)，缺失 key 返回 None，不抛 KeyError
    #   Config 没有 .get() 方法（__getattr__ 对非配置 key 抛 AttributeError）
    #   因此不能用 config.get("key", default)，要用 config["key"] or default
    report = {
        "timestamp":       datetime.now().strftime("%Y%m%d_%H%M%S"),
        "retrieval_model": config["retrieval_model_path"] or "N/A",
        "rerank_model":    config["rerank_model_path"] or "N/A",    # or 处理 None
        "generator_model": config["generator_model"] or "N/A",
        "retrieval_topk":  config["retrieval_topk"],
        "rerank_topk":     config["rerank_topk"] or "N/A",          # or 处理 None
        # 将 [0,1] 的分数转为百分比
        "scores":          {k: round(v * 100, 2) for k, v in scores.items()},
    }
    if extra_info:
        report.update(extra_info)

    save_dir = config["save_dir"]
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"report_{report['timestamp']}.json")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"  报告已保存: {save_path}")
    return report  # Dict: 完整评估报告


print("端到端函数定义完毕")

### 10.2 一键运行完整流程

In [ ]:
# ============================================================
# 用户配置区（仅需修改此处）
# ============================================================
E2E_WORKSPACE    = "flashrag_e2e"                        # str: 工作目录
E2E_DATASET_NAME = "mydata"                               # str: 数据集名
E2E_API_KEY      = os.environ.get("DEEPSEEK_API_KEY", "your-key")  # str: API 密钥
E2E_BASE_URL     = "https://api.deepseek.com/v1"         # str: API 地址
E2E_EMBED_MODEL  = "BAAI/bge-small-zh-v1.5"             # str: 向量编码模型
E2E_RERANK_MODEL = "BAAI/bge-reranker-base"              # str: 重排序模型
E2E_GEN_MODEL    = "deepseek-chat"                       # str: 生成模型

# ---- 原始文档（替换为你自己的文档）----
e2e_raw_texts = [
    {"title": "RAG 概述",
     "content": "RAG 即检索增强生成，结合检索和大模型，解决幻觉和知识更新问题。"
                "三阶段：朴素 RAG（检索生成）、高级 RAG（重排序/改写）、模块化 RAG。"},
    {"title": "FlashRAG",
     "content": "FlashRAG 是中国人民大学开源 RAG 框架，内置 Naive RAG、Self-RAG、FLARE 等算法，"
                "支持 OpenaiGenerator、HFCausalLMGenerator，以及 DenseRetriever 和 BM25Retriever。"},
    {"title": "高级 RAG",
     "content": "FLARE 在生成低置信度 token 时主动检索，将检索和生成交织进行。"
                "Self-RAG 通过反思 token 自主决定是否检索。"},
]

# ---- 问答测试集（替换为你自己的问题）----
e2e_qa = [
    {"id": "1", "question": "RAG 的三个发展阶段是什么？",
     "golden_answers": ["朴素 RAG、高级 RAG、模块化 RAG"]},
    {"id": "2", "question": "FlashRAG 支持哪些生成器后端？",
     "golden_answers": ["OpenaiGenerator、HFCausalLMGenerator"]},
    {"id": "3", "question": "FLARE 算法的核心特点？",
     "golden_answers": ["低置信度 token 时主动检索"]},
]

# ============================================================
# 完整端到端流程（6 步）
# ============================================================
e2e_corpus_path = f"{E2E_WORKSPACE}/corpus/corpus.jsonl"
e2e_index_dir   = f"{E2E_WORKSPACE}/index"

print("[Step 1/6] 文档分块 → 写入 JSONL")
n = step1_build_corpus(e2e_raw_texts, e2e_corpus_path)
print(f"   → {n} 个 chunk")

print("\n[Step 2/6] 构建 FAISS 索引（Index_Builder）")
e2e_index_path = step2_build_index(
    e2e_corpus_path, e2e_index_dir,
    model_path=E2E_EMBED_MODEL, retrieval_method="e5"
)
print(f"   → 索引文件: {e2e_index_path}")

print("\n[Step 3/6] 创建 Config")
e2e_config = step3_setup_config(
    workspace       = E2E_WORKSPACE,
    dataset_name    = E2E_DATASET_NAME,
    corpus_path     = e2e_corpus_path,
    index_path      = e2e_index_path,
    api_key         = E2E_API_KEY,
    base_url        = E2E_BASE_URL,
    generator_model = E2E_GEN_MODEL,
    retrieval_model = E2E_EMBED_MODEL,
    rerank_model    = E2E_RERANK_MODEL,
)
print(f"   → dataset_path={e2e_config['dataset_path']}")

print("\n[Step 4/6] 准备数据集")
e2e_dataset = step4_prepare_dataset(e2e_qa, E2E_WORKSPACE, E2E_DATASET_NAME, e2e_config)
print(f"   → {len(e2e_dataset)} 条问答")

print("\n[Step 5/6] 运行 Pipeline（检索 → 重排序 → 生成）")
# 初始化重排序器（CrossReranker，来自官方源码）
e2e_reranker = CrossReranker(e2e_config)  # CrossReranker: 官方类名
e2e_output   = step5_run_pipeline(e2e_config, e2e_dataset, e2e_reranker)

print("\n[Step 6/6] 评估并保存报告")
e2e_report = step6_evaluate(
    e2e_config, e2e_output,
    extra_info={"experiment": "e2e_rerank_rag"}
)

# ---- 打印汇总 ----
print("\n" + "=" * 55)
print("端到端流程完成！评估结果:")
for metric, score in e2e_report["scores"].items():
    print(f"  {metric:<12}: {score:.2f}%")

print("\n问题预测对照:")
for item in e2e_output:
    print(f"  Q: {item['question']}")
    print(f"  A: {item['pred'][:80]}...")
    print(f"  ✓: {item['golden_answers']}")
    print()

### 10.3 组件替换速查表

所有组件可独立替换，其余代码无需修改：

| 替换目标 | 修改位置 | 示例 |
|----------|---------|------|
| 换检索模型 | `step3_setup_config(retrieval_model=...)` | `"BAAI/bge-large-zh-v1.5"` |
| 换重排序模型 | `step3_setup_config(rerank_model=...)` | `"cross-encoder/ms-marco-MiniLM-L-6-v2"` |
| 换生成模型 | `step3_setup_config(generator_model=...)` | `"gpt-4o-mini"` |
| 换本地生成 | `step3_setup_config(framework="hf")` | 需配合 `generator_model_path` |
| 换 BM25 检索 | `retrieval_method="bm25"` | `bm25_backend="bm25s"` |
| 关闭重排序 | `use_reranker=False` | 删除 `step5` 中的 `rerank` 调用 |
| 换 Pipeline 算法 | 替换 `step5` 的 `BasicPipeline` 子类 | `FLAREPipeline`、`SelfRAGPipeline` |

## 十一、总结

### 本 Notebook 知识点回顾

| 章节 | 核心内容 | 关键 API（均经官方源码验证） |
|------|---------|----------------------------|
| 3.1 | YAML 配置 | `Config(path, override_dict)` |
| 4.1 | 文档分块 | `split_text_into_chunks()` → JSONL |
| 4.2 | 构建索引 | `Index_Builder(...).build_index()` |
| 4.3 | 加载检索器 | `DenseRetriever(config)`（索引须预先构建） |
| 5.1 | 稠密检索 | `retriever.batch_search(queries)` |
| 5.2 | BM25 检索 | `BM25Retriever(config).batch_search(queries)` |
| 5.3 | 混合检索 | `use_multi_retriever=True` → `MultiRetrieverRouter` |
| 6 | 重排序 | `CrossReranker(config).rerank(queries, docs_list)` → `(docs, scores)` |
| 7.1 | API 生成 | `OpenaiGenerator(config).generate(input_list)` |
| 7.2 | 本地生成 | `get_generator(config)` → `HFCausalLMGenerator` |
| 7.3 | Prompt 模板 | `PromptTemplate(config).get_string(question=q, retrieval_result=r)` |
| 8.2 | Naive RAG | `SequentialPipeline(config).run(dataset)` |
| 8.3 | FLARE | `FLAREPipeline(config).run(dataset)`（需开启 logprobs） |
| 8.4 | HyDE（手动） | `generate(hypo_prompt)` → `batch_search(hypo_doc)` |
| 8.5 | 自定义 Pipeline | 继承 `BasicPipeline`，重写 `run()` |
| 9 | 评估 | `Evaluator(config).evaluate(dataset)` |
| **10** | **端到端** | **step1→step6 串联，一键跑通** |

### 常见 API 误区（官方源码与常见错误对照）

| ❌ 错误写法 | ✓ 正确写法 | 来源依据 |
|------------|-----------|----------|
| `retriever.retrieve(queries)` | `retriever.batch_search(queries)` | retriever.py 源码 |
| `from flashrag.retriever.reranker import CrossEncoderReranker` | `from flashrag.retriever import CrossReranker` | reranker.py 源码 |
| `docs = reranker.rerank(q, docs)` | `docs, scores = reranker.rerank([q], [docs])` | reranker.py 源码 |
| `"bm25_backend": "rank_bm25"` | `"bm25_backend": "bm25s"` | retriever.py 源码 |
| `template.build_prompt(q, docs)` | `template.get_string(question=q, retrieval_result=docs)` | pipeline.py 源码 |
| `HyDEPipeline` | 不存在，手动实现 | pipeline/__init__.py 源码 |
| `HFGenerator` | `HFCausalLMGenerator`（或 `get_generator()`） | utils.py 源码 |
| `DenseRetriever` 自动建索引 | 必须先用 `Index_Builder.build_index()` | retriever.py 源码 |
| `BM25Retriever` 自动建索引 | 同上，`bm25s.BM25.load()` 只加载，不构建 | retriever.py 源码 |
| 索引文件扩展名 `.faiss` | 实际为 `.index`（`{method}_{type}.index`） | index_builder.py 源码 |
| BM25 index_path 为文件路径 | 实际为目录路径 `{save_dir}/bm25/` | index_builder.py 源码 |

### RAG 系统建设核心清单

```
□ 1. 语料准备：PDF/HTML → 纯文本 → split_text_into_chunks → JSONL
□ 2. 索引构建：Index_Builder(...).build_index()
     Dense: 生成 {save_dir}/{method}_{faiss_type}.index
     BM25:  生成 {save_dir}/bm25/ 目录（Dense 和 BM25 都需要预构建！）
□ 3. 配置管理：data_dir+dataset_name → Config（data_dir/dataset_name/test.jsonl）
□ 4. 检索策略：Dense（语义）/ BM25（关键词）/ 多路混合（两者兼顾）
□ 5. 重排序  ：CrossReranker.rerank() 返回 (docs, scores) 元组
□ 6. 生成后端：API（OpenaiGenerator）/ 本地（get_generator() → HFCausalLMGenerator）
□ 7. Pipeline ：SequentialPipeline / FLAREPipeline / 自定义继承 BasicPipeline
□ 8. 评估指标：Evaluator（EM/F1/acc，来自 basic_config.yaml 默认 metrics）
```